# TF-IDF Cluster Labeling

Top discriminating terms per cluster, from `cleaned_text`.
Doc-level TF-IDF, weights averaged per cluster.

In [1]:
import pandas as pd
import numpy as np
import json
import re
import unicodedata
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Load

In [2]:
df = pd.read_csv("tovima_clustered.csv", encoding="utf-32", sep="\t")
df["to_lists"] = df["to_lists"].apply(json.loads)
df["to_other_recipients"] = df["to_other_recipients"].apply(json.loads)

labeled = df[df["final_label"] != -1].copy().reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Labeled rows: {len(labeled)}")
print(f"Clusters: {labeled['final_label'].nunique()}")
print(f"Noise rows: {(df['final_label'] == -1).sum()}")

Total rows: 13637
Labeled rows: 11626
Clusters: 107
Noise rows: 2011


## 2. Text preparation

Exclude list applied at text level + NFD accent stripping.

In [3]:
EXCLUDE_TERMS = {
    # normalize_patterns() placeholders
    "ημερομηνια", "ωρα", "τηλεφωνο", "αιθουσα", "μαθημα",
    # structural corpus noise
    "tovima", "announcements",
    # English boilerplate
    "the", "and", "of", "university",
    # generic verbs
    "μπορω", "μπορώ",
}

EXCLUDE_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in EXCLUDE_TERMS) + r")\b",
    re.IGNORECASE,
)


def remove_greek_accents(text):
    nfd = unicodedata.normalize("NFD", text)
    # Mn = combining accent marks
    result = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    return result.replace("\u0345", "")  # iota subscript


def clean_for_tfidf(text):
    if not isinstance(text, str):
        return ""
    text = EXCLUDE_PATTERN.sub(" ", text)
    text = remove_greek_accents(text)
    return re.sub(r"\s+", " ", text).strip()


labeled["tfidf_text"] = labeled["cleaned_text"].apply(clean_for_tfidf)

# spot-check
for i in [0, 100, 500]:
    print(f"Row {i}:")
    print(f"  before: {labeled['cleaned_text'].iloc[i][:120]!r}")
    print(f"  after:  {labeled['tfidf_text'].iloc[i][:120]!r}")
    print()

Row 0:
  before: 'announcements ανακοίνωση θέση phd ελβετία επισυνάπτεται ανακοίνωση θέση phd ελβετία'
  after:  'ανακοινωση θεση phd ελβετια επισυναπτεται ανακοινωση θεση phd ελβετια'

Row 100:
  before: 'tovima διοργάνωση διεθνούς συνεδρίου μαθημα αθήνα παρών επιστολή θέλω ανακοινώσω διοργάνωση 18th european conference'
  after:  'διοργανωση διεθνους συνεδριου αθηνα παρων επιστολη θελω ανακοινωσω διοργανωση 18th european conference composite materia'

Row 500:
  before: 'tovima παρθενική συνεδρίαση επιτροπής συγχώνευσης παν πατρών τει δυτ ελλάδος ενημερωτικό'
  after:  'παρθενικη συνεδριαση επιτροπης συγχωνευσης παν πατρων τει δυτ ελλαδος ενημερωτικο'



## 3. Fit TF-IDF

min_df=5, max_df=0.6, tokens >= 4 chars, sublinear TF, unigrams + bigrams.

In [4]:
texts = labeled["tfidf_text"].tolist()
labels = labeled["final_label"].tolist()

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.6,
    token_pattern=r"(?u)\b\w{4,}\b",
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)
terms = np.array(vectorizer.get_feature_names_out())

print(f"Vocabulary size: {len(terms):,}")
print(f"Document-term matrix shape: {X.shape}")

Vocabulary size: 64,088
Document-term matrix shape: (11626, 64088)


## 4. Filter bigram fragments

In [5]:
def is_valid_term(term):
    parts = term.split()
    return all(len(p) >= 4 for p in parts)


valid_mask = np.array([is_valid_term(t) for t in terms])
print(f"Terms passing fragment filter: {valid_mask.sum():,} / {len(terms):,}")
print(f"Filtered out: {(~valid_mask).sum():,} fragment bigrams")

Terms passing fragment filter: 64,088 / 64,088
Filtered out: 0 fragment bigrams


## 5. Score terms per cluster

In [6]:
TOP_N = 15  # terms to extract per cluster (show 15, use top 10 for labeling)

cluster_ids = sorted(set(labels))
top_terms = {}
term_weights = {}

for cid in cluster_ids:
    idx = [i for i, l in enumerate(labels) if l == cid]
    cluster_matrix = X[idx]

    # mean weight per term
    mean_weights = np.asarray(cluster_matrix.mean(axis=0)).ravel()

    # zero out invalid terms
    mean_weights[~valid_mask] = 0

    sorted_idx = mean_weights.argsort()[::-1]
    cluster_top = [
        (terms[j], round(float(mean_weights[j]), 5))
        for j in sorted_idx
        if mean_weights[j] > 0
    ][:TOP_N]

    top_terms[cid] = [t for t, _ in cluster_top]
    term_weights[cid] = cluster_top

print(f"Computed top terms for {len(top_terms)} clusters")

Computed top terms for 107 clusters


## 6. Inspect results

In [7]:
cluster_sizes = labeled["final_label"].value_counts().to_dict()

print(f"{'Cluster':>8}  {'Size':>5}  Top 10 terms")
print("-" * 100)
for cid in sorted(cluster_ids, key=lambda c: -cluster_sizes.get(c, 0)):
    size = cluster_sizes.get(cid, 0)
    terms_str = " | ".join(top_terms[cid][:10])
    print(f"{cid:>8}  {size:>5}  {terms_str}")

 Cluster   Size  Top 10 terms
----------------------------------------------------------------------------------------------------
      26    938  παρουσιαση | διπλωματικης | εργασιας | μεταπτυχιακης | παρουσιαση μεταπτυχιακης | μεταπτυχιακης διπλωματικης | εργασια | διπλωματικης εργασιας | καθηγητης | επιβλεπων
      17    819  διατριβης | διδακτορικης | διδακτορικης διατριβης | υποστηριξη | υποστηριξη διδακτορικης | παρουσιαση | διατριβη | δημοσια | τμηματος | διδακτορικος
      23    390  συλλυπητηρια | συλλυπητηρια ανακοινωση | απωλεια | εκφραζω | θλιψη | οικογενεια | ανακοινωση | θερμος | τμηματος | θερμος συλλυπητηρια
      74    329  χημικος | ομιλητης | chemical | engineering | μηχανικος | τμημα χημικος | χημικος μηχανικος | ομιλιας | research | τιτλος ομιλιας
      40    324  μεταπτυχιακων | σπουδων | μεταπτυχιακων σπουδων | αιτηση | γραμματεια τμηματος | προγραμμα μεταπτυχιακων | διπλωματος | προγραμμα | τμηματος | προκηρυξη
      97    308  απεργια | συγκεντρωση | εργαζομεν

## 7. Sample emails per cluster

In [8]:
N_SAMPLE = 4

print("=" * 100)
for cid in sorted(cluster_ids, key=lambda c: -cluster_sizes.get(c, 0)):
    size = cluster_sizes.get(cid, 0)
    terms_preview = " | ".join(top_terms[cid][:8])
    print(f"\nCluster {cid} ({size} emails)")
    print(f"  Top terms: {terms_preview}")
    cluster_rows = labeled[labeled["final_label"] == cid]
    for subj in cluster_rows["Subject"].sample(
        min(N_SAMPLE, len(cluster_rows)), random_state=42
    ):
        print(f"  • {subj[:95]}")


Cluster 26 (938 emails)
  Top terms: παρουσιαση | διπλωματικης | εργασιας | μεταπτυχιακης | παρουσιαση μεταπτυχιακης | μεταπτυχιακης διπλωματικης | εργασια | διπλωματικης εργασιας
  • [TOVIMA] Παρουσίαση διπλωματικής εργασίας κ. Χρίστίνας Ντότσικα
  • [TOVIMA] Παρουσίαση διπλωματικής εργασίας του φοιτητή του ΤΗΜκΤΥ κ. Δ. Κορμπου
  • [TOVIMA] δημόσια παρουσίαση ΜΔΕ κας Χ. Παπανικολάου
  • [TOVIMA] ΠΑΡΟΥΣΙΑΣΗ ΔΙΠΛΩΜΑΤΙΚΗΣ ΕΡΓΑΣΙΑΣ

Cluster 17 (819 emails)
  Top terms: διατριβης | διδακτορικης | διδακτορικης διατριβης | υποστηριξη | υποστηριξη διδακτορικης | παρουσιαση | διατριβη | δημοσια
  • [ANNOUNCEMENTS] Δημόσια υποστήριξη Διδακτορικής Διατριβής Υ.Δ. Τμ. Πολιτικών Μηχανικών κα
  • [TOVIMA] ΥΠΕΝΘΥΜΙΣΗ. ΔΗΜΟΣΙΑ ΥΠΟΣΤΗΡΙΞΗ ΔΙΔΑΚΤΟΡΙΚΗΣ ΔΙΑΤΡΙΒΗΣΤΗΣ ΥΠΟΨΗΦΙΑΣ ΔΙΔΑΚΤΟΡΟΣ ΤΟΥ ΤΜΗΜ
  • [TOVIMA] Δημόσια παρουσίαση Διδακτορικής Διατριβής
  • [TOVIMA] ΟΡΘΗ ΕΠΑΝΑΛΗΨΗ_Δημόσια υποστήριξη της διδακτορικής διατριβής του ΥΔ Νικόλα Χούρι

Cluster 23 (390 emails)
  Top te

## 8. Save top-terms output

In [9]:
# detailed: (cluster, term)
rows = []
for cid in sorted(cluster_ids, key=lambda c: -cluster_sizes.get(c, 0)):
    for rank, (term, weight) in enumerate(term_weights[cid], 1):
        rows.append({
            "cluster_id": cid,
            "cluster_size": cluster_sizes.get(cid, 0),
            "rank": rank,
            "term": term,
            "mean_tfidf_weight": weight,
        })

terms_df = pd.DataFrame(rows)
terms_df.to_csv("cluster_top_terms.csv", index=False, encoding="utf-8-sig")
print(f"Saved cluster_top_terms.csv: {len(terms_df)} rows")

# compact: one row per cluster
compact = []
for cid in sorted(cluster_ids, key=lambda c: -cluster_sizes.get(c, 0)):
    compact.append({
        "cluster_id": cid,
        "cluster_size": cluster_sizes.get(cid, 0),
        "top_10_terms": " | ".join(top_terms[cid][:10]),
    })

compact_df = pd.DataFrame(compact)
compact_df.to_csv("cluster_labels_compact.csv", index=False, encoding="utf-8-sig")
print(f"Saved cluster_labels_compact.csv: {len(compact_df)} rows")

Saved cluster_top_terms.csv: 1605 rows
Saved cluster_labels_compact.csv: 107 rows


## 9. Merge labels into the full dataset

Noise gets major_topic = specific_topic = "Noise" instead of NaN.

In [10]:
labels_df = pd.read_csv("cluster_labels_final.csv", encoding="utf-8-sig")

# keep needed columns
labels_df = labels_df[["cluster_id", "major_topic", "specific_topic"]].copy()

# merge on final_label = cluster_id
df_full = pd.read_csv("tovima_clustered.csv", encoding="utf-32", sep="\t")
df_full = df_full.merge(
    labels_df,
    left_on="final_label",
    right_on="cluster_id",
    how="left",
)
df_full = df_full.drop(columns=["cluster_id"])

df_full["major_topic"] = df_full["major_topic"].fillna("Noise")
df_full["specific_topic"] = df_full["specific_topic"].fillna("Noise")

print(f"Shape: {df_full.shape}")
print(f"Columns: {df_full.columns.tolist()}")
print()
print("major_topic distribution:")
print(df_full["major_topic"].value_counts().to_string())

Shape: (13637, 20)
Columns: ['Author', 'Date', 'To', 'Subject', 'Message', 'Date_parsed', 'Year', 'had_html_markup', 'is_reply_or_forward', 'to_lists', 'to_other_recipients', 'subject_tag', 'contains_pii_pattern', 'embedding_text', 'cleaned_text', 'cluster_label', 'merged_label', 'final_label', 'major_topic', 'specific_topic']

major_topic distribution:
major_topic
Academic Events                       3383
Noise                                 2011
Campus Life                           1573
University Administration             1443
Research & Funding                     942
Labour & Union Affairs                 932
Elections & Governance                 886
Postgraduate Programmes                509
Health & COVID                         465
Informal Discussion                    444
Innovation & Entrepreneurship          257
University Rankings & Distinctions     193
Miscellaneous                          179
Digital Infrastructure                 168
Continuing Education          

In [11]:
# sanity check
assert df_full["major_topic"].isna().sum() == 0, "Some rows have null major_topic"
assert df_full["specific_topic"].isna().sum() == 0, "Some rows have null specific_topic"
print("\nSanity checks passed.")


Sanity checks passed.


In [12]:
df_out = df_full.copy()
df_out["to_lists"] = df_out["to_lists"].apply(
    lambda x: x if isinstance(x, str) else json.dumps(x)
)
df_out["to_other_recipients"] = df_out["to_other_recipients"].apply(
    lambda x: x if isinstance(x, str) else json.dumps(x)
)

df_out.to_csv("tovima_final.csv", sep="\t", encoding="utf-32", index=False)
print(f"Saved tovima_final.csv: {df_out.shape}")

Saved tovima_final.csv: (13637, 20)
